# Tests de integración de `filter_flows()`

Notebook orientado a verificar el comportamiento end-to-end de OP-12 `Filter flows`,
considerando resultado tabular, resultado operacional y trazabilidad.

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- imports generales,
- imports del módulo,
- helpers de testing reutilizables,
- y configuración básica de display.

### 0.1 Imports generales

In [1]:
from copy import deepcopy

import pandas as pd
import h3

### 0.2 Imports del módulo

In [2]:
from pylondrina.datasets import FlowDataset
from pylondrina.errors import FilterError
from pylondrina.transforms.flows_filtering import FlowFilterOptions, filter_flows

### 0.3 Helpers de testing reutilizables

In [3]:
def h3_from_latlon(lat: float, lon: float, res: int = 8) -> str:
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, res)
    return h3.geo_to_h3(lat, lon, res)


def issues_to_df(issues):
    return pd.DataFrame(
        [
            {
                "level": issue.level,
                "code": issue.code,
                "message": issue.message,
                "field": issue.field,
                "source_field": issue.source_field,
                "row_count": issue.row_count,
                "details": issue.details,
            }
            for issue in issues
        ]
    )


def assert_issue_codes(
    issues,
    expected_present=(),
    expected_absent=(),
):
    codes = [issue.code for issue in issues]
    for code in expected_present:
        assert code in codes, f"Falta issue {code}. Codes emitidos: {codes}"
    for code in expected_absent:
        assert code not in codes, f"No debía aparecer {code}. Codes emitidos: {codes}"


def get_last_event(flows: FlowDataset) -> dict:
    events = flows.metadata.get("events", [])
    assert isinstance(events, list) and len(events) > 0, "No hay eventos en metadata['events']"
    return events[-1]


def snapshot_flowdataset_state(flows: FlowDataset) -> dict:
    return {
        "flows": flows.flows.copy(deep=True),
        "flow_to_trips": None if flows.flow_to_trips is None else flows.flow_to_trips.copy(deep=True),
        "aggregation_spec": deepcopy(flows.aggregation_spec),
        "metadata": deepcopy(flows.metadata),
        "provenance": deepcopy(flows.provenance),
    }


def assert_flowdataset_input_intact(flows: FlowDataset, snapshot: dict) -> None:
    pd.testing.assert_frame_equal(flows.flows, snapshot["flows"])
    if snapshot["flow_to_trips"] is None:
        assert flows.flow_to_trips is None
    else:
        pd.testing.assert_frame_equal(flows.flow_to_trips, snapshot["flow_to_trips"])
    assert flows.aggregation_spec == snapshot["aggregation_spec"]
    assert flows.metadata == snapshot["metadata"]
    assert flows.provenance == snapshot["provenance"]

### 0.4 Configuración de display

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

### 0.5 Fixtures ricos de FlowDataset

In [5]:
def _build_base_flows_df():
    origin_a = h3_from_latlon(-33.4500, -70.6500, 8)
    origin_b = h3_from_latlon(-33.4400, -70.6400, 8)
    origin_c = h3_from_latlon(-33.4600, -70.6600, 8)
    origin_d = h3_from_latlon(-33.4700, -70.6700, 8)

    dest_a = h3_from_latlon(-33.4300, -70.6300, 8)
    dest_b = h3_from_latlon(-33.4200, -70.6200, 8)
    dest_c = h3_from_latlon(-33.4100, -70.6100, 8)

    flows_df = pd.DataFrame(
        {
            "flow_id": ["f01", "f02", "f03", "f04", "f05", "f06", "f07", "f08", "f09", "f10", "f11", "f12"],
            "origin_h3_index": [origin_a, origin_a, origin_a, origin_b, origin_b, origin_b, origin_c, origin_c, origin_d, origin_d, origin_a, origin_c],
            "destination_h3_index": [dest_a, dest_b, dest_c, dest_a, dest_b, origin_a, dest_b, dest_c, dest_a, dest_b, origin_a, origin_a],
            "flow_count": [12, 7, 3, 9, 2, 1, 15, 5, 4, 6, 8, 11],
            "flow_value": [18.0, 7.0, 4.0, 20.0, 2.0, 5.0, 30.0, 8.0, 4.0, 12.0, 8.0, 50.0],
            "mode": ["bus", "metro", "walk", "bus", "bike", "bus", "car", "bus", "metro", "bus", "scooter", "bus"],
            "purpose": ["work", "work", "education", "work", "leisure", "health", "work", "leisure", "education", "work", "shopping", "work"],
            "gender": ["F", "M", "F", "M", "F", None, "M", "F", "F", "F", "M", "F"],
            "day_type": ["weekday", "weekday", "weekday", "weekday", "weekend", "weekday", "weekday", "weekend", "weekday", "weekday", "weekend", "weekday"],
            "time_period": ["am_peak", "am_peak", "am_peak", "am_peak", "midday", "midday", "pm_peak", "pm_peak", "am_peak", "am_peak", "midday", "pm_peak"],
            "corridor": ["north", "north", "north", "west", "west", "west", "south", "south", "east", "east", "north", "south"],
            "window_start_utc": pd.to_datetime(
                [
                    "2026-01-01T10:00:00Z",
                    "2026-01-01T10:30:00Z",
                    "2026-01-01T11:00:00Z",
                    "2026-01-01T12:00:00Z",
                    "2026-01-02T15:00:00Z",
                    "2026-01-02T16:00:00Z",
                    "2026-01-03T22:00:00Z",
                    "2026-01-03T22:30:00Z",
                    "2026-01-04T09:00:00Z",
                    "2026-01-04T09:15:00Z",
                    "2026-01-04T17:00:00Z",
                    "2026-01-05T23:00:00Z",
                ],
                utc=True,
            ),
            "window_end_utc": pd.to_datetime(
                [
                    "2026-01-01T10:40:00Z",
                    "2026-01-01T11:10:00Z",
                    "2026-01-01T11:20:00Z",
                    "2026-01-01T12:45:00Z",
                    "2026-01-02T15:20:00Z",
                    "2026-01-02T16:10:00Z",
                    "2026-01-03T22:30:00Z",
                    "2026-01-03T22:50:00Z",
                    "2026-01-04T09:20:00Z",
                    "2026-01-04T09:55:00Z",
                    "2026-01-04T17:25:00Z",
                    "2026-01-05T23:40:00Z",
                ],
                utc=True,
            ),
        }
    )

    cells = {
        "origin_a": origin_a,
        "origin_b": origin_b,
        "origin_c": origin_c,
        "origin_d": origin_d,
        "dest_a": dest_a,
        "dest_b": dest_b,
        "dest_c": dest_c,
    }

    return flows_df, cells


def _build_flow_to_trips_df():
    rows = []
    n_links_by_flow = {
        "f01": 3,
        "f02": 2,
        "f03": 1,
        "f04": 3,
        "f05": 1,
        "f06": 1,
        "f07": 4,
        "f08": 2,
        "f09": 1,
        "f10": 2,
        "f11": 2,
        "f12": 3,
    }
    for flow_id, n_links in n_links_by_flow.items():
        for i in range(1, n_links + 1):
            rows.append({"flow_id": flow_id, "movement_id": f"m_{flow_id}_{i:02d}"})
    return pd.DataFrame(rows)


def _base_metadata(is_validated: bool = False):
    return {
        "dataset_id": "flows_rich_demo",
        "artifact_id": "artifact_rich_demo",
        "is_validated": bool(is_validated),
        "events": [
            {"op": "build_flows", "ts_utc": "2026-04-07T10:00:00Z"},
            {"op": "write_flows", "ts_utc": "2026-04-07T10:10:00Z"},
        ],
        "h3": {"resolution": 8},
        "custom_tag": "integration_fixture",
    }


def _base_aggregation_spec():
    return {
        "h3_resolution": 8,
        "group_by": ["mode", "purpose", "gender", "day_type", "time_period"],
    }


def _base_provenance():
    return {
        "source": "synthetic_manual_fixture",
        "note": "integration tests OP-12",
    }


def make_flowdataset_segmented_rich():
    flows_df, cells = _build_base_flows_df()
    ds = FlowDataset(
        flows=flows_df,
        flow_to_trips=None,
        aggregation_spec=_base_aggregation_spec(),
        source_trips=None,
        metadata=_base_metadata(is_validated=False),
        provenance=_base_provenance(),
    )
    return ds, cells


def make_flowdataset_with_trip_links_rich():
    flows_df, cells = _build_base_flows_df()
    flow_to_trips_df = _build_flow_to_trips_df()
    ds = FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips_df,
        aggregation_spec=_base_aggregation_spec(),
        source_trips=None,
        metadata=_base_metadata(is_validated=False),
        provenance=_base_provenance(),
    )
    return ds, cells


def make_flowdataset_validated_rich():
    flows_df, cells = _build_base_flows_df()
    flow_to_trips_df = _build_flow_to_trips_df()
    ds = FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips_df,
        aggregation_spec=_base_aggregation_spec(),
        source_trips=None,
        metadata=_base_metadata(is_validated=True),
        provenance=_base_provenance(),
    )
    return ds, cells

### 0.6 Inspección rápida de fixtures

In [6]:
flowdataset_segmented_rich, cells_segmented = make_flowdataset_segmented_rich()
flowdataset_with_trip_links_rich, cells_with_links = make_flowdataset_with_trip_links_rich()
flowdataset_validated_rich, cells_validated = make_flowdataset_validated_rich()

display(flowdataset_segmented_rich.flows.head())
display(flowdataset_with_trip_links_rich.flow_to_trips.head())
print(flowdataset_segmented_rich.flows.shape)
print(flowdataset_with_trip_links_rich.flow_to_trips.shape)
print(flowdataset_validated_rich.metadata)

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,gender,day_type,time_period,corridor,window_start_utc,window_end_utc
0,f01,88b2c554e9fffff,88b2c556a1fffff,12,18.0,bus,work,F,weekday,am_peak,north,2026-01-01 10:00:00+00:00,2026-01-01 10:40:00+00:00
1,f02,88b2c554e9fffff,88b2c55685fffff,7,7.0,metro,work,M,weekday,am_peak,north,2026-01-01 10:30:00+00:00,2026-01-01 11:10:00+00:00
2,f03,88b2c554e9fffff,88b2c55689fffff,3,4.0,walk,education,F,weekday,am_peak,north,2026-01-01 11:00:00+00:00,2026-01-01 11:20:00+00:00
3,f04,88b2c554cdfffff,88b2c556a1fffff,9,20.0,bus,work,M,weekday,am_peak,west,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
4,f05,88b2c554cdfffff,88b2c55685fffff,2,2.0,bike,leisure,F,weekend,midday,west,2026-01-02 15:00:00+00:00,2026-01-02 15:20:00+00:00


,flow_id,movement_id
0,f01,m_f01_01
1,f01,m_f01_02
2,f01,m_f01_03
3,f02,m_f02_01
4,f02,m_f02_02


(12, 13)
(25, 2)
{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'is_validated': True, 'events': [{'op': 'build_flows', 'ts_utc': '2026-04-07T10:00:00Z'}, {'op': 'write_flows', 'ts_utc': '2026-04-07T10:10:00Z'}], 'h3': {'resolution': 8}, 'custom_tag': 'integration_fixture'}


## Sección 1. Integration tests de filter_flows
### Test 1 - caso principal correcto: combinación where + h3_cells con auxiliar sincronizado

Qué prueba: el camino feliz principal sobre un fixture rico, verificando output tabular, OperationReport, summary, parameters, evento, flow_to_trips sincronizado, preservación de is_validated y no mutación del input.


In [7]:
flows, cells = make_flowdataset_with_trip_links_rich()
snapshot = snapshot_flowdataset_state(flows)

options = FlowFilterOptions(
    where={
        "mode": ["bus", "metro"],
        "purpose": "work",
        "flow_value": {"gte": 10},
        "gender": "F",
    },
    h3_cells=[cells["origin_a"], cells["origin_d"]],
    spatial_predicate="origin",
    keep_flow_to_trips=True,
    keep_metadata=True,
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
    max_issues=50,
)

# Output tabular
assert filtered is not flows
assert filtered.flows["flow_id"].tolist() == ["f01", "f10"]

# Summary
assert report.ok is True
assert report.summary["rows_in"] == 12
assert report.summary["rows_out"] == 2
assert report.summary["dropped_total"] == 10
assert report.summary["dropped_by_filter"] == {"where": 9, "h3_cells": 1}
assert report.summary["filters_requested"] == ["where", "h3_cells"]
assert report.summary["filters_applied"] == ["where", "h3_cells"]
assert report.summary["filters_omitted"] == []
assert report.summary["flow_to_trips_status"] == "synced"

# Parameters
assert report.parameters["spatial_predicate"] == "origin"
assert report.parameters["keep_flow_to_trips"] is True
assert report.parameters["keep_metadata"] is True
assert report.parameters["strict"] is False
assert report.parameters["max_issues"] == 50

# flow_to_trips sincronizado
assert filtered.flow_to_trips is not None
assert sorted(filtered.flow_to_trips["flow_id"].unique().tolist()) == ["f01", "f10"]

# Metadata / evento
assert filtered.metadata["is_validated"] is False
last_event = get_last_event(filtered)
assert last_event["op"] == "filter_flows"
assert last_event["parameters"] == report.parameters
assert last_event["summary"] == report.summary
assert "issues_summary" in last_event
assert "context" not in last_event

# aggregation_spec preservada
assert filtered.aggregation_spec == flows.aggregation_spec

# provenance derivada
assert filtered.provenance is not None
assert filtered.provenance["derived_from"][0]["dataset_id"] == flows.metadata["dataset_id"]
assert filtered.provenance["derived_from"][0]["artifact_id"] == flows.metadata["artifact_id"]
assert "prior_events_summary" in filtered.provenance

# Input no mutado
assert_flowdataset_input_intact(flows, snapshot)

display(filtered.flows)
display(filtered.flow_to_trips)
display(issues_to_df(report.issues))

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,gender,day_type,time_period,corridor,window_start_utc,window_end_utc
0,f01,88b2c554e9fffff,88b2c556a1fffff,12,18.0,bus,work,F,weekday,am_peak,north,2026-01-01 10:00:00+00:00,2026-01-01 10:40:00+00:00
9,f10,88b2c555dbfffff,88b2c55685fffff,6,12.0,bus,work,F,weekday,am_peak,east,2026-01-04 09:15:00+00:00,2026-01-04 09:55:00+00:00


,flow_id,movement_id
0,f01,m_f01_01
1,f01,m_f01_02
2,f01,m_f01_03
18,f10,m_f10_01
19,f10,m_f10_02


,level,code,message,field,source_field,row_count,details
0,info,FLT_FLOW.WHERE.APPLIED,Se aplicó el eje `where` sobre FlowDataset.flows.,None,None,9,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
1,info,FLT_FLOW.H3.APPLIED,Se aplicó el eje `h3_cells` sobre `origin_h3_index` / `destination_h3_index`.,None,None,1,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
2,info,FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED,Se filtró `flow_to_trips` para mantener consistencia con los `flow_id` retenidos.,None,None,20,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."


### Test 2 - caso principal espacial: spatial_predicate="both" sobre fixture segmentado

Qué prueba: semántica espacial both sobre el contrato nuevo de flows, sin flow_to_trips, y con resultado múltiple pero controlado.

In [8]:
flows, cells = make_flowdataset_segmented_rich()
snapshot = snapshot_flowdataset_state(flows)

options = FlowFilterOptions(
    h3_cells=[cells["origin_a"], cells["dest_a"], cells["dest_b"]],
    spatial_predicate="both",
    keep_flow_to_trips=True,
    keep_metadata=True,
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
)

assert filtered.flows["flow_id"].tolist() == ["f01", "f02", "f11"]
assert report.summary["rows_in"] == 12
assert report.summary["rows_out"] == 3
assert report.summary["filters_requested"] == ["h3_cells"]
assert report.summary["filters_applied"] == ["h3_cells"]
assert report.summary["filters_omitted"] == []
assert report.summary["flow_to_trips_status"] == "missing"

assert_issue_codes(
    report.issues,
    expected_present=[
        "FLT_FLOW.H3.APPLIED",
        "FLT_FLOW.AUX.FLOW_TO_TRIPS_REQUESTED_BUT_MISSING",
    ],
)

last_event = get_last_event(filtered)
assert last_event["summary"] == report.summary
assert last_event["parameters"] == report.parameters

assert_flowdataset_input_intact(flows, snapshot)

display(filtered.flows)
display(issues_to_df(report.issues))

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,gender,day_type,time_period,corridor,window_start_utc,window_end_utc
0,f01,88b2c554e9fffff,88b2c556a1fffff,12,18.0,bus,work,F,weekday,am_peak,north,2026-01-01 10:00:00+00:00,2026-01-01 10:40:00+00:00
1,f02,88b2c554e9fffff,88b2c55685fffff,7,7.0,metro,work,M,weekday,am_peak,north,2026-01-01 10:30:00+00:00,2026-01-01 11:10:00+00:00
10,f11,88b2c554e9fffff,88b2c554e9fffff,8,8.0,scooter,shopping,M,weekend,midday,north,2026-01-04 17:00:00+00:00,2026-01-04 17:25:00+00:00


,level,code,message,field,source_field,row_count,details
0,info,FLT_FLOW.H3.APPLIED,Se aplicó el eje `h3_cells` sobre `origin_h3_index` / `destination_h3_index`.,None,None,9.0,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': False, 'h3..."
1,info,FLT_FLOW.AUX.FLOW_TO_TRIPS_REQUESTED_BUT_MISSING,"Se solicitó conservar `flow_to_trips`, pero el auxiliar no está presente; el resultado se devolverá con `flow_to_tri...",None,None,NaN,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': False, 'h3..."


### Test 3 - fatal/precondición: falta columna canónica mínima y aborta sin mutar el input

Qué prueba: abort fatal de contrato/precondición antes del pipeline normal.

In [9]:
flows, _ = make_flowdataset_segmented_rich()
snapshot = snapshot_flowdataset_state(flows)

flows_bad = FlowDataset(
    flows=flows.flows.drop(columns=["flow_value"]).copy(deep=True),
    flow_to_trips=flows.flow_to_trips,
    aggregation_spec=deepcopy(flows.aggregation_spec),
    source_trips=flows.source_trips,
    metadata=deepcopy(flows.metadata),
    provenance=deepcopy(flows.provenance),
)

raised = None
try:
    filter_flows(flows_bad, options=FlowFilterOptions(where={"mode": "bus"}))
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, FilterError)
assert getattr(raised, "code", None) == "FLT_FLOW.CONTRACT.MISSING_CANONICAL_COLUMNS"

# El input original usado para construir la variante no debe mutarse.
assert_flowdataset_input_intact(flows, snapshot)

### Test 4 - caso degradado con strict=False: where inválido pero H3 sí aplicable

Qué prueba: degradación controlada con issue error retornable, sin abortar, aplicando el otro eje del filtro.

In [10]:
flows, cells = make_flowdataset_segmented_rich()
snapshot = snapshot_flowdataset_state(flows)

options = FlowFilterOptions(
    where={"campo_inexistente": "x"},
    h3_cells=[cells["origin_b"]],
    spatial_predicate="origin",
    keep_flow_to_trips=True,
    keep_metadata=True,
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
)

assert report.ok is False
assert filtered.flows["flow_id"].tolist() == ["f04", "f05", "f06"]

assert report.summary["filters_requested"] == ["where", "h3_cells"]
assert report.summary["filters_applied"] == ["h3_cells"]
assert report.summary["filters_omitted"] == ["where"]
assert report.summary["flow_to_trips_status"] == "missing"

assert_issue_codes(
    report.issues,
    expected_present=[
        "FLT_FLOW.WHERE.FIELD_MISSING",
        "FLT_FLOW.H3.APPLIED",
        "FLT_FLOW.AUX.FLOW_TO_TRIPS_REQUESTED_BUT_MISSING",
    ],
)

last_event = get_last_event(filtered)
assert last_event["summary"] == report.summary
assert last_event["parameters"] == report.parameters

assert_flowdataset_input_intact(flows, snapshot)

display(filtered.flows)
display(issues_to_df(report.issues))

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,gender,day_type,time_period,corridor,window_start_utc,window_end_utc
3,f04,88b2c554cdfffff,88b2c556a1fffff,9,20.0,bus,work,M,weekday,am_peak,west,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
4,f05,88b2c554cdfffff,88b2c55685fffff,2,2.0,bike,leisure,F,weekend,midday,west,2026-01-02 15:00:00+00:00,2026-01-02 15:20:00+00:00
5,f06,88b2c554cdfffff,88b2c554e9fffff,1,5.0,bus,health,None,weekday,midday,west,2026-01-02 16:00:00+00:00,2026-01-02 16:10:00+00:00


,level,code,message,field,source_field,row_count,details
0,error,FLT_FLOW.WHERE.FIELD_MISSING,El eje `where` no puede evaluar el campo 'campo_inexistente' porque no existe en FlowDataset.flows; esa regla se omi...,campo_inexistente,None,12.0,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
1,info,FLT_FLOW.H3.APPLIED,Se aplicó el eje `h3_cells` sobre `origin_h3_index` / `destination_h3_index`.,None,None,9.0,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
2,info,FLT_FLOW.AUX.FLOW_TO_TRIPS_REQUESTED_BUT_MISSING,"Se solicitó conservar `flow_to_trips`, pero el auxiliar no está presente; el resultado se devolverá con `flow_to_tri...",None,None,NaN,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."


### Test 5 - metadata / event / summary consistente en fixture “validado”

Qué prueba: que la operación preserve is_validated=True, preserve aggregation_spec, rehaga provenance como derivada y mantenga consistencia exacta entre report.summary, report.parameters y el evento final.

In [11]:
flows, _ = make_flowdataset_validated_rich()
snapshot = snapshot_flowdataset_state(flows)
n_events_before = len(flows.metadata["events"])

options = FlowFilterOptions(
    where={
        "mode": "bus",
        "flow_count": {"gte": 8},
    },
    keep_flow_to_trips=True,
    keep_metadata=True,
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
)

assert filtered.flows["flow_id"].tolist() == ["f01", "f04", "f12"]

# Preserva exactamente is_validated
assert filtered.metadata["is_validated"] is True

# Evento append-only
assert len(filtered.metadata["events"]) == n_events_before + 1
last_event = get_last_event(filtered)
assert last_event["op"] == "filter_flows"
assert last_event["summary"] == report.summary
assert last_event["parameters"] == report.parameters
assert "issues_summary" in last_event

# aggregation_spec intacta
assert filtered.aggregation_spec == flows.aggregation_spec

# provenance derivada y no simple copia pasiva
assert filtered.provenance is not None
assert filtered.provenance["derived_from"][0]["dataset_id"] == flows.metadata["dataset_id"]
assert filtered.provenance["derived_from"][0]["artifact_id"] == flows.metadata["artifact_id"]
assert "prior_events_summary" in filtered.provenance
assert isinstance(filtered.provenance["prior_events_summary"], list)

assert_flowdataset_input_intact(flows, snapshot)

display(filtered.flows)
display(issues_to_df(report.issues))
display(pd.DataFrame(filtered.metadata["events"]))

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,gender,day_type,time_period,corridor,window_start_utc,window_end_utc
0,f01,88b2c554e9fffff,88b2c556a1fffff,12,18.0,bus,work,F,weekday,am_peak,north,2026-01-01 10:00:00+00:00,2026-01-01 10:40:00+00:00
3,f04,88b2c554cdfffff,88b2c556a1fffff,9,20.0,bus,work,M,weekday,am_peak,west,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
11,f12,88b2c554e1fffff,88b2c554e9fffff,11,50.0,bus,work,F,weekday,pm_peak,south,2026-01-05 23:00:00+00:00,2026-01-05 23:40:00+00:00


,level,code,message,field,source_field,row_count,details
0,info,FLT_FLOW.WHERE.APPLIED,Se aplicó el eje `where` sobre FlowDataset.flows.,None,None,9,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
1,info,FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED,Se filtró `flow_to_trips` para mantener consistencia con los `flow_id` retenidos.,None,None,16,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."


,op,ts_utc,parameters,summary,issues_summary
0,build_flows,2026-04-07T10:00:00Z,NaN,NaN,NaN
1,write_flows,2026-04-07T10:10:00Z,NaN,NaN,NaN
2,filter_flows,2026-04-07T20:28:46.953587Z,"{'where': {'mode': 'bus', 'flow_count': {'gte': 8}}, 'h3_cells': None, 'spatial_predicate': 'origin', 'keep_flow_to_...","{'rows_in': 12, 'rows_out': 3, 'dropped_total': 9, 'dropped_by_filter': {'where': 9, 'h3_cells': 0}, 'filters_reques...","{'counts': {'info': 2, 'warning': 0, 'error': 0}, 'top_codes': [{'code': 'FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED', 'count..."


### Test 6 - caso de política clave: keep_metadata=False

Qué prueba: política cerrada de metadata mínima: sin historial/evento nuevo, pero preservando metadata operativa e is_validated.

In [12]:
flows, _ = make_flowdataset_validated_rich()
snapshot = snapshot_flowdataset_state(flows)

options = FlowFilterOptions(
    where={
        "mode": ["bus", "scooter"],
        "day_type": "weekend",
    },
    keep_flow_to_trips=True,
    keep_metadata=False,
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
)

assert filtered.flows["flow_id"].tolist() == ["f08", "f11"]

# No historial/evento nuevo
assert "events" not in filtered.metadata

# Metadata operativa mínima preservada
assert filtered.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert filtered.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert filtered.metadata["is_validated"] is True
assert filtered.metadata["h3"] == flows.metadata["h3"]
assert filtered.metadata["custom_tag"] == flows.metadata["custom_tag"]

# report sigue completo
assert report.parameters is not None
assert report.summary["rows_in"] == 12
assert report.summary["rows_out"] == 2

assert_flowdataset_input_intact(flows, snapshot)

display(filtered.flows)
display(issues_to_df(report.issues))

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,gender,day_type,time_period,corridor,window_start_utc,window_end_utc
7,f08,88b2c554e1fffff,88b2c55689fffff,5,8.0,bus,leisure,F,weekend,pm_peak,south,2026-01-03 22:30:00+00:00,2026-01-03 22:50:00+00:00
10,f11,88b2c554e9fffff,88b2c554e9fffff,8,8.0,scooter,shopping,M,weekend,midday,north,2026-01-04 17:00:00+00:00,2026-01-04 17:25:00+00:00


,level,code,message,field,source_field,row_count,details
0,info,FLT_FLOW.WHERE.APPLIED,Se aplicó el eje `where` sobre FlowDataset.flows.,None,None,10,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
1,info,FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED,Se filtró `flow_to_trips` para mantener consistencia con los `flow_id` retenidos.,None,None,21,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."


### Test 7 - caso de política clave: flow_to_trips inválido se descarta explícitamente

Qué prueba: política cerrada del auxiliar: si existe pero no es usable, se descarta a None con evidencia, sin romper el filtrado principal.

In [ ]:
flows, _ = make_flowdataset_with_trip_links_rich()
snapshot = snapshot_flowdataset_state(flows)

flows_bad_aux = FlowDataset(
    flows=flows.flows.copy(deep=True),
    flow_to_trips=flows.flow_to_trips.drop(columns=["movement_id"]).copy(deep=True),
    aggregation_spec=deepcopy(flows.aggregation_spec),
    source_trips=flows.source_trips,
    metadata=deepcopy(flows.metadata),
    provenance=deepcopy(flows.provenance),
)

options = FlowFilterOptions(
    where={"mode": "bus"},
    keep_flow_to_trips=True,
    keep_metadata=True,
    strict=False,
)

filtered, report = filter_flows(
    flows_bad_aux,
    options=options,
)

assert filtered.flows["flow_id"].tolist() == ["f01", "f04", "f06", "f08", "f10", "f12"]
assert filtered.flow_to_trips is None
assert report.summary["flow_to_trips_status"] == "discarded_invalid"
assert report.ok is False

assert_issue_codes(
    report.issues,
    expected_present=[
        "FLT_FLOW.WHERE.APPLIED",
        "FLT_FLOW.AUX.FLOW_TO_TRIPS_INVALID",
    ],
)

last_event = get_last_event(filtered)
assert last_event["summary"] == report.summary
assert last_event["parameters"] == report.parameters

# El fixture original sigue intacto
assert_flowdataset_input_intact(flows, snapshot)

display(filtered.flows)
display(issues_to_df(report.issues))

### Test 8 - política clave: strict=True con error recuperable escalar a FilterError

Qué prueba: escalamiento correcto cuando hay issue error recuperable y strict=True.

In [13]:
flows, cells = make_flowdataset_segmented_rich()
snapshot = snapshot_flowdataset_state(flows)

raised = None
try:
    filter_flows(
        flows,
        options=FlowFilterOptions(
            where={"campo_inexistente": "x"},
            h3_cells=[cells["origin_a"]],
            spatial_predicate="origin",
            keep_flow_to_trips=True,
            keep_metadata=True,
            strict=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, FilterError)
assert raised.issue is not None
assert raised.issue.code == "FLT_FLOW.WHERE.FIELD_MISSING"
assert raised.issues is not None
assert any(issue.code == "FLT_FLOW.WHERE.FIELD_MISSING" for issue in raised.issues)

# Input no mutado
assert_flowdataset_input_intact(flows, snapshot)

display(issues_to_df(raised.issues))

,level,code,message,field,source_field,row_count,details
0,error,FLT_FLOW.WHERE.FIELD_MISSING,El eje `where` no puede evaluar el campo 'campo_inexistente' porque no existe en FlowDataset.flows; esa regla se omi...,campo_inexistente,None,12.0,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': True, 'where_provided': True, 'h3_c..."
1,info,FLT_FLOW.H3.APPLIED,Se aplicó el eje `h3_cells` sobre `origin_h3_index` / `destination_h3_index`.,None,None,8.0,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': True, 'where_provided': True, 'h3_c..."
2,info,FLT_FLOW.AUX.FLOW_TO_TRIPS_REQUESTED_BUT_MISSING,"Se solicitó conservar `flow_to_trips`, pero el auxiliar no está presente; el resultado se devolverá con `flow_to_tri...",None,None,NaN,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': True, 'where_provided': True, 'h3_c..."


### Test 9 - caso degradado con warning: resultado vacío retornable

Qué prueba: resultado vacío como warning retornable, sin abortar, manteniendo contrato de salida y evento.

In [14]:
flows, _ = make_flowdataset_with_trip_links_rich()
snapshot = snapshot_flowdataset_state(flows)

options = FlowFilterOptions(
    where={
        "mode": "bus",
        "flow_count": {"lt": 0},
    },
    keep_flow_to_trips=True,
    keep_metadata=True,
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
)

assert filtered.flows.empty is True
assert report.ok is True
assert report.summary["rows_in"] == 12
assert report.summary["rows_out"] == 0
assert report.summary["dropped_total"] == 12
assert report.summary["flow_to_trips_status"] == "synced"

assert_issue_codes(
    report.issues,
    expected_present=[
        "FLT_FLOW.WHERE.APPLIED",
        "FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED",
        "FLT_FLOW.RESULT.EMPTY_DATASET",
    ],
)

last_event = get_last_event(filtered)
assert last_event["summary"] == report.summary
assert last_event["parameters"] == report.parameters

assert_flowdataset_input_intact(flows, snapshot)

display(filtered.flows)
display(filtered.flow_to_trips)
display(issues_to_df(report.issues))

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,gender,day_type,time_period,corridor,window_start_utc,window_end_utc


,flow_id,movement_id


,level,code,message,field,source_field,row_count,details
0,info,FLT_FLOW.WHERE.APPLIED,Se aplicó el eje `where` sobre FlowDataset.flows.,None,None,12,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
1,info,FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED,Se filtró `flow_to_trips` para mantener consistencia con los `flow_id` retenidos.,None,None,25,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
2,warning,FLT_FLOW.RESULT.EMPTY_DATASET,El filtrado produjo un FlowDataset vacío.,None,None,12,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."


### Test 10 - truncamiento de issues + bloque limits + consistencia evento/report

Qué prueba: control de tamaño del reporte y consistencia del bloque limits en summary y evento.

In [15]:
flows, _ = make_flowdataset_segmented_rich()
snapshot = snapshot_flowdataset_state(flows)

options = FlowFilterOptions(
    where={
        "does_not_exist": "x",
        "mode": {"gt": "bus"},
        "window_start_utc": {"gte": "bad_ts"},
        "gender": {"is_null": False},
    },
    keep_flow_to_trips=True,
    keep_metadata=True,
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
    max_issues=3,
)

assert "limits" in report.summary
assert report.summary["limits"]["max_issues"] == 3
assert report.summary["limits"]["issues_truncated"] is True
assert report.summary["limits"]["n_issues_emitted"] <= 3
assert report.summary["limits"]["n_issues_detected_total"] >= report.summary["limits"]["n_issues_emitted"]

assert_issue_codes(
    report.issues,
    expected_present=["FLT_FLOW.REPORT.ISSUES_TRUNCATED"],
)

last_event = get_last_event(filtered)
assert last_event["summary"] == report.summary
assert last_event["parameters"] == report.parameters
assert "issues_summary" in last_event

assert_flowdataset_input_intact(flows, snapshot)

display(filtered.flows)
display(issues_to_df(report.issues))

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,gender,day_type,time_period,corridor,window_start_utc,window_end_utc
0,f01,88b2c554e9fffff,88b2c556a1fffff,12,18.0,bus,work,F,weekday,am_peak,north,2026-01-01 10:00:00+00:00,2026-01-01 10:40:00+00:00
1,f02,88b2c554e9fffff,88b2c55685fffff,7,7.0,metro,work,M,weekday,am_peak,north,2026-01-01 10:30:00+00:00,2026-01-01 11:10:00+00:00
2,f03,88b2c554e9fffff,88b2c55689fffff,3,4.0,walk,education,F,weekday,am_peak,north,2026-01-01 11:00:00+00:00,2026-01-01 11:20:00+00:00
3,f04,88b2c554cdfffff,88b2c556a1fffff,9,20.0,bus,work,M,weekday,am_peak,west,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
4,f05,88b2c554cdfffff,88b2c55685fffff,2,2.0,bike,leisure,F,weekend,midday,west,2026-01-02 15:00:00+00:00,2026-01-02 15:20:00+00:00
5,f06,88b2c554cdfffff,88b2c554e9fffff,1,5.0,bus,health,None,weekday,midday,west,2026-01-02 16:00:00+00:00,2026-01-02 16:10:00+00:00
6,f07,88b2c554e1fffff,88b2c55685fffff,15,30.0,car,work,M,weekday,pm_peak,south,2026-01-03 22:00:00+00:00,2026-01-03 22:30:00+00:00
7,f08,88b2c554e1fffff,88b2c55689fffff,5,8.0,bus,leisure,F,weekend,pm_peak,south,2026-01-03 22:30:00+00:00,2026-01-03 22:50:00+00:00
8,f09,88b2c555dbfffff,88b2c556a1fffff,4,4.0,metro,education,F,weekday,am_peak,east,2026-01-04 09:00:00+00:00,2026-01-04 09:20:00+00:00
9,f10,88b2c555dbfffff,88b2c55685fffff,6,12.0,bus,work,F,weekday,am_peak,east,2026-01-04 09:15:00+00:00,2026-01-04 09:55:00+00:00


,level,code,message,field,source_field,row_count,details
0,error,FLT_FLOW.WHERE.FIELD_MISSING,El eje `where` no puede evaluar el campo 'does_not_exist' porque no existe en FlowDataset.flows; esa regla se omitirá.,does_not_exist,None,12.0,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
1,error,FLT_FLOW.WHERE.OPERATOR_INCOMPATIBLE,El operador 'gt' no es compatible con el campo 'mode' bajo el contrato vigente de OP-12; esa regla se omitirá.,mode,None,12.0,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': True, 'h3_..."
2,warning,FLT_FLOW.REPORT.ISSUES_TRUNCATED,La lista de issues de filter_flows fue truncada por `max_issues=3`.,None,None,NaN,"{'max_issues': 3, 'n_issues_detected_total': 6, 'n_issues_emitted': 3, 'issues_truncated': True, 'reason': 'max_issu..."


### Test 11 - política explícita: options=None produce “sin filtros definidos” pero retorna dataset derivado válido

Qué prueba: caso borde “sin filtros definidos”, con dataset nuevo, mismo contenido tabular, evento y preservación de estado.

In [16]:
flows, _ = make_flowdataset_with_trip_links_rich()
snapshot = snapshot_flowdataset_state(flows)

filtered, report = filter_flows(
    flows,
    options=None,
    max_issues=20,
)

assert filtered is not flows
pd.testing.assert_frame_equal(filtered.flows, flows.flows)
pd.testing.assert_frame_equal(filtered.flow_to_trips, flows.flow_to_trips)

assert report.ok is True
assert report.summary["rows_in"] == 12
assert report.summary["rows_out"] == 12
assert report.summary["dropped_total"] == 0
assert report.summary["filters_requested"] == []
assert report.summary["filters_applied"] == []
assert report.summary["filters_omitted"] == []
assert report.summary["flow_to_trips_status"] == "synced"

assert_issue_codes(
    report.issues,
    expected_present=[
        "FLT_FLOW.NO_CHANGES.NO_FILTERS_DEFINED",
        "FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED",
    ],
)

last_event = get_last_event(filtered)
assert last_event["summary"] == report.summary
assert last_event["parameters"] == report.parameters

assert filtered.metadata["is_validated"] is False

assert_flowdataset_input_intact(flows, snapshot)

display(filtered.flows.head())
display(issues_to_df(report.issues))

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,gender,day_type,time_period,corridor,window_start_utc,window_end_utc
0,f01,88b2c554e9fffff,88b2c556a1fffff,12,18.0,bus,work,F,weekday,am_peak,north,2026-01-01 10:00:00+00:00,2026-01-01 10:40:00+00:00
1,f02,88b2c554e9fffff,88b2c55685fffff,7,7.0,metro,work,M,weekday,am_peak,north,2026-01-01 10:30:00+00:00,2026-01-01 11:10:00+00:00
2,f03,88b2c554e9fffff,88b2c55689fffff,3,4.0,walk,education,F,weekday,am_peak,north,2026-01-01 11:00:00+00:00,2026-01-01 11:20:00+00:00
3,f04,88b2c554cdfffff,88b2c556a1fffff,9,20.0,bus,work,M,weekday,am_peak,west,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
4,f05,88b2c554cdfffff,88b2c55685fffff,2,2.0,bike,leisure,F,weekend,midday,west,2026-01-02 15:00:00+00:00,2026-01-02 15:20:00+00:00


,level,code,message,field,source_field,row_count,details
0,info,FLT_FLOW.NO_CHANGES.NO_FILTERS_DEFINED,No se definieron filtros efectivos; OP-12 finalizó sin cambios efectivos.,None,None,NaN,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': False, 'h3..."
1,info,FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED,Se filtró `flow_to_trips` para mantener consistencia con los `flow_id` retenidos.,None,None,0.0,"{'dataset_id': 'flows_rich_demo', 'artifact_id': 'artifact_rich_demo', 'strict': False, 'where_provided': False, 'h3..."
